# Trabajo Final de Machine Learning - Ejercicio 1
## Regresion: resistencia a la compresion del hormigon

**Grupo:** completar
**Integrantes:** completar

Este notebook compara tres algoritmos supervisados para predecir la resistencia del hormigon en MPa. Incluye metadata, EDA, preparacion, entrenamiento, evaluacion e interpretacion.

## 1. Metadata y planteamiento

El dataset de UCI contiene 1.030 mezclas de hormigon, ocho variables predictoras cuantitativas y la resistencia de compresion como variable objetivo. Las cantidades de componentes estan expresadas en kg/m3, la edad en dias y la salida en MPa.

La pregunta es: **dada una mezcla y su edad, que resistencia de compresion podemos estimar?**

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='deep')
RANDOM_STATE = 42

In [ ]:
DATA_PATH = Path('/Users/alexis/Downloads/concrete+compressive+strength/Concrete_Data.xls')
if not DATA_PATH.exists():
    DATA_PATH = Path('Concrete_Data.xls')

df = pd.read_excel(DATA_PATH)
df.columns = ['cement', 'slag', 'fly_ash', 'water', 'superplasticizer', 'coarse_aggregate', 'fine_aggregate', 'age_days', 'strength_mpa']
print(f'Dimensiones: {df.shape}')
display(df.head())

In [ ]:
display(df.describe().T)
print('Valores faltantes:', df.isna().sum().sum())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['strength_mpa'], kde=True, ax=axes[0], color='#1976a5')
axes[0].set_title('Distribucion de resistencia')
sns.heatmap(df.corr(numeric_only=True), cmap='vlag', center=0, ax=axes[1])
axes[1].set_title('Correlaciones')
plt.tight_layout()
plt.show()

## 2. Preparacion y modelos

Separamos un conjunto de prueba estratificado de forma aleatoria. La estandarizacion se coloca dentro de un pipeline para evitar fuga de informacion durante la validacion cruzada.

Modelos: regresion lineal, Random Forest y Gradient Boosting.

In [ ]:
X = df.drop(columns='strength_mpa')
y = df['strength_mpa']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)

models = {
    'Regresion lineal': Pipeline([('scale', StandardScaler()), ('model', LinearRegression())]),
    'Random Forest': RandomForestRegressor(n_estimators=350, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.04, max_depth=2, random_state=RANDOM_STATE)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({'Modelo': name, 'MAE (MPa)': mean_absolute_error(y_test, pred), 'RMSE (MPa)': mean_squared_error(y_test, pred) ** 0.5, 'R2': r2_score(y_test, pred)})
results_df = pd.DataFrame(results).sort_values('RMSE (MPa)')
display(results_df.style.format({'MAE (MPa)':'{:.2f}', 'RMSE (MPa)':'{:.2f}', 'R2':'{:.3f}'}))

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in models.items():
    scores = -cross_val_score(model, X, y, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
    cv_rows.append({'Modelo': name, 'RMSE CV promedio': scores.mean(), 'Desvio CV': scores.std()})
display(pd.DataFrame(cv_rows).sort_values('RMSE CV promedio').style.format({'RMSE CV promedio':'{:.2f}', 'Desvio CV':'{:.2f}'}))

In [ ]:
best_name = results_df.iloc[0]['Modelo']
best_model = models[best_name]
best_pred = best_model.predict(X_test)
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=best_pred, color='#e76f51', alpha=0.75)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
plt.plot(lims, lims, '--', color='black')
plt.xlabel('Resistencia real (MPa)')
plt.ylabel('Resistencia predicha (MPa)')
plt.title(f'Real vs predicho - {best_name}')
plt.show()

if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=X.columns).sort_values()
elif 'model' in best_model.named_steps and hasattr(best_model.named_steps['model'], 'coef_'):
    importance = pd.Series(abs(best_model.named_steps['model'].coef_), index=X.columns).sort_values()
else:
    importance = pd.Series(dtype=float)
if not importance.empty:
    importance.plot.barh(figsize=(8, 5), color='#2a9d8f', title=f'Importancia de variables - {best_name}')
    plt.xlabel('Importancia absoluta / relativa')
    plt.show()

## 3. Interpretacion y conclusiones

- El modelo recomendado es el que presenta menor RMSE en el conjunto de prueba y un comportamiento consistente en validacion cruzada.
- La regresion lineal funciona como linea base interpretable; los modelos de ensamble pueden capturar relaciones no lineales entre edad, agua y componentes.
- El grafico real vs. predicho permite verificar sesgo y dispersion. Los puntos alejados de la diagonal representan mezclas cuya resistencia es mas dificil de estimar.
- La importancia de variables indica que atributos contribuyen mas a la prediccion, pero no implica causalidad.

**Limitaciones:** el resultado depende de una unica particion de prueba, de la calidad del dataset historico y no debe sustituir ensayos de laboratorio ni criterios de ingenieria estructural.